INFERENCE PIPELINE: GENERAZIONE DI TESTO CONDIZIONATO.

L'AI non è solo un entità libera di sognare testi, ma nel mondo reale, se chiedo un output al computer, voglio che sia prevedibile, controllato e veloce (testo condizionato).

Si intende il processo con cui un modello già addestrato riceve un certo contesto e genera testo condizionato da quel contesto.
La parola importante è condizionato.
Un modello autoregressivo non genera testo dal nulla. Genera il prossimo token in funzione dei token che h già ricevuto

La generazione non è libera, è condizionata da istruzioni, documenti, eventuali esempi, eventuali dati, eventuale adapter LoRA
Solo dopo aver dato il contesto parte la generazione.

- Grammar Constraints: forzre l LLM a generare output strutturati come JSON. Come obbligare un modello probabilistico a parlare nel linguaggio rigido del software, come per esempio JSON.
- Stop Sequences: controllare la terminazione per evitare l'uso inutile di calcolo. L'arte di dire 'basta' alla generazione, prima che diventi uno spreco di risorse.
- Streaming: ottimizzazione della latenza percepita tramite emissione asincrona di token. Cercando di eliminare la frutazione dell'utente mentre aspetta la risposta. Inviando i pensieri della macchina mentre si formano.

Grammar Constraints
Dalla prosa libera alla struttura rigida.
Forzare la creatività dentro uno schema rigido
Gli LLM sono spesso addestrati per essere creativi, noi abbiamo spesso bisogno che siano noiosi e precisi. Se integrato un modello in un app, una virgol fuori posto in un file JSON può far creschare il sistma. E' qui che intervengono i vincoli grammaticali.
Un limite intrinseco degli LLM è la loro natura probabilistica (addestrai per essere creativi), che può portare a violazioni strutturali quando integriamo il modello in pipeline software.
I vincoli grammaticali intervengono sul campionamento dei token per garantire che l'output segue uno schema predefinito, come un oggetto JSON valido.

Ma come facciamo, matematicamente, a impedire alla rete di sbagliare?
La magia avviene a livello di Logit Masking
Prima che il modello scelga la parola successiva, intervniamo sulla sua classifica di probabilità, usiamo un analizzatore sintattico, che sa esattamente cosa è permesso in quel momento. Se il modello sta scrivendo un JSON, ed ha appena aperto una parentesi graffa, azzeriamo tutti i token che non siano una virgoletta o uno spazio. Forziamo il modello a popolare chiavi specifiche con tipi di dato corretti.
Filtrare la distribuzione dei token.
- Contraint Sampling: tecnica che azzera la probabilità dei token che non rispettano lo stato attuale della grammatica
- Analisi Sinteattica: il sistema mantiene un automa a stati finiti che definisce quali caratteri sono ammessi dopo la sequenza corrente.

Fortunatamente non dobbiamo scrivere noi a mano questi automatismi ogni volta.
Efficienza dei Vincoli
Esistono librerie nate appositamente per questo scopo, come 'outlines' o 'guidance' che permettono di definire schemi Pydantic o una regex, ed usali come guida per l'inferenza senza ri-addestramento.
La cosa straordinaria è che questo controllo ha un overhead quasi nullo. Il mascheramento avviene durante la fase di selezione del token, aggiungendo una latenza trascurabile rispetto alla generazione standard (zero overhead).
Garantire la struttura non è solo un vezzo estetico, ma riduce drasticamente gli errori di runtime nelle applicazioni che consumano i dati generati (robustezza delle API).

Grazie a questo LLM smette di essere un semplice narratore ma diventa anche un motore di generazione dati affidabile e deterministico.

Una volta che il modello ha iniziato a generare correttamente, come facciamo a farlo smettere?
Definire i confini della generazione
Senza un segnale di stop chiaro, il modello potrebbe continuare a generare testo inutile o ripetitivo fino al raggiungimento del limite di token. 
Le Stop Sequenes sono stringe specifiche che, se rilvate, interrompono immediatamente il proceso di inferenza, risparmiando tempo e costi.

Quali sono i segnali per interrompere il flusso?
Esistono segnali standard e segnali personalizzati. Il token EOS (End of Sequence) è il punto fermo che il modello ha imparato a usare durante il traninng per chiudere un discorso, la fine naturale di un testo.
Spesso abbiamo bisogno di più controllo, in un'applicazione di chat dobbiamo interrompere il modello dopo "user:" ed aspettare la risposta.
Algoritmi di ricerca come il beam search possono fermarsi quando tutti i rami attivi raggiungono una sequenza di stop (Early Stopping)
L'uso corretto dello stop impedisce la generazione di 'code' indesiderate che sporcano l'output finale dell'applicazione.

Ma cosa succede se il modello entra in loop infinito?
Abbiamo bisogno di un paracadute
Il parametro Max Tokens è il nostro paracadute hardware. Indipendentemente da cosa il modello voglia dire, quando raggiungiamo quel limite la generazione si ferma.
Ferma la generazione tempestivamente ottimizza il throuput del server, permettendoci di servire più utenti contemporaneamente. 
Monitorare il buffer di output è un compito che spetta al server di inferenza per identificare lo stop sequences prima di inviare i dati al client.
Fermare la generazione non appena il compito è completato ottimizza il throughput e riduce i costi per i modelli a consumo.

Ma possiamo anche incoraggiare il modello a finire prima, intervenendo sulla sua psicologia statistica

Calibrazione della fine
Spesso i modelli tendono a ritardare la fine della frase per massimizzare la probabilità cumulativa, portando a prolissità, perchè magari la statistica li spinge a cercare la frase perfetta anche se questa è lunga
Intervenire sulla probabilità del token EOS tramite penalità o premi può forzare il modello a essere più conciso ed efficiente.
E' come dire al modello "apprezzerei molto su tu arrivassi al punto velocemente".

Ma c'è un problema: anche la risposta più breve può sembrare eterna se l'utente deve attendere che sia completata tutta quanta prima di poterla leggere. E' è qui che entra in gioco lo streaming

Streaming
Vincere la latenza del primo token
E' come se vesse le parole comparire sulla carta mentre vegono pensate.
Generare un paragrafo può richiedere 5 o 10 secondi, in termini di user experience questi secondo sono un eternità tanto da poterli spingere ad abbandonare la chat. Lo streaming invece trasforma l'attesa in una lettura dinamica e coinvolgente.

Usiamo una tecnologia chiamata Server-Sent-Events (SSE)
Architettura della risposta asincrona
Invece di aspettare il pacchetto completo il modello produce un token, lo decodifica e lo invia immediatamente lungo il canale socket (iterative Decoding).
La metrica che ci interessa non è più il tempo totale ma il TTFT (time to firt token), che misura quanto tempo l'utente attende prima di vedere l'inizio della risposta. Anche se per finire la fra se ci vorrà altro tempo.
La velocità di elaborazione degli LLm moderni è ormai simile alla velocità di letetura umana, creando un'illusione di fluidità perfetta.
 
Implementazione Tecnica.
Librerie come Hugging Face offrono classi dedicate per stampare a console o inviare via API i token in tempo reale (TextStreamer).
Il server deve gestire i buffer parziali per assicurarsi che i caratteri multi-byte o i token spezzati siano visualizzati correttametne (gestione della coda).
Dobbiamo quindi gestire un buffer intelligente che si svuota solo quando il carattere è completo.
Inoltre, fare post-processing: esempio censura di un testo o formattazione di un testo che sta ancora arrivando è una sfida tecnica non banale, che richiede calcoli costanti.

Ma questo streming ha un costo nascosto?

Confrontiamo le prestazioni
Throughput vs Latenza
C'è sempre un compromesso
Lo straming può leggermente diminuire il throughput massimo del server a causa dell'overhead dei pacchetti di rete frequenti (anzichè un singolo pacchetto grande).
Tuttavia, il guadagno in termini di engagement e soddisfazione utente rende lo streaming lo standard 'de facto' per le interfacce conversazionali.
Preferiamo sacrificare un po di efficienza nel software per guadagnare la soddisfazione dell'utente

Training e inference sono due fasi diverse
Durante il training hai:
inputt -> modello -> output previsto -> confronto con risposta corretta -> loss -> backpropagation
Durante inference hai:
input -> modello già addestrato -> generate () -> output
Non hai loss.backward() e optimizer.step()
I pesi non vengono modificati.
Quindi
durante il training -> impara
durante inference -> usa ciò che è stato appreso.

Durante inference puoi caricare un adapter LoRA sopra il modello base usando PeftModel.from_pretrained(...) e ppuoi anche avere più adapter e selezionare quello attivo.
Quindi, per esempio, posso averee:
LoRA commerciale
LoRA tecnico
LoRA qualità
e scegliere l'adapter giusto prima della generazione.

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer, LogitsProcessor, LogitsProcessorList, StoppingCriteria, StoppingCriteriaList

# 1. CONFIGURAZIONE DEL MODELLO E DEL TOKENIZER
# Inizializziamo il modello GPT-2, un sistema di linguaggio pre-addestrato ottimizzato per la generazione di testo.
# Il tokenizer ha il compito di convertire il testo in sequenze numeriche (token) elaborabili dal modello.
# AutoModelForCausalLM carica i pesi e l'architettura necessaria per prevedere il token successivo.
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 2. DEFINIZIONE DEI VINCOLI GRAMMATICALI: FILTRO PER LE LETTERE MAIUSCOLE
# Questa classe eredita da LogitsProcessor e interviene direttamente nel processo decisionale del modello.
class NoUppercaseProcessor(LogitsProcessor):
    """
    Il processore modifica la distribuzione di probabilità (logits) prima che venga scelto il token successivo.
    L'obiettivo è forzare il modello a evitare qualsiasi parola che contenga lettere maiuscole.
    """
    def __call__(self, input_ids, scores):
        # Analizziamo l'intero dizionario dei token disponibili per identificare quelli non desiderati.
        for token_id in range(len(scores[0])):
            # Convertiamo l'ID numerico del token in testo per verificarne la capitalizzazione.
            token_text = tokenizer.decode([token_id])
            
            # Se il token contiene almeno un carattere maiuscolo, assegniamo un punteggio di meno infinito.
            # Questo rende impossibile la selezione di quel token durante la fase di campionamento.
            if any(char.isupper() for char in token_text):
                scores[:, token_id] = float('-inf') 
        return scores

# 3. DEFINIZIONE DEI CRITERI DI STOP: INTERRUZIONE AL PUNTO FERMO
# Questa classe eredita da StoppingCriteria e definisce una condizione logica di uscita.
class StopAtPeriodCriteria(StoppingCriteria):
    """
    Il criterio monitora il flusso di generazione e invia un segnale di arresto quando viene prodotto un punto.
    Questo meccanismo è essenziale per garantire che l'output si fermi in modo naturale a fine frase.
    """
    def __call__(self, input_ids, scores, **kwargs):
        # Recuperiamo l'ultimissimo token generato dal modello in questo passaggio.
        last_token = tokenizer.decode(input_ids[0][-1])
        
        # Se il token contiene un punto fermo, restituiamo True per terminare immediatamente il ciclo.
        return "." in last_token

# 4. PREPARAZIONE DELLA GENERAZIONE IN STREAMING
# Definiamo la frase di partenza (prompt) e la trasformiamo in tensori PyTorch comprensibili dal modello.
prompt = "Some thoughts about technology, Apple vs Microsoft:"
inputs = tokenizer(prompt, return_tensors="pt")

# TextStreamer gestisce la visualizzazione progressiva del testo sul terminale (token per token).
# Questo approccio riduce la latenza percepita, permettendo di leggere l'output mentre viene creato.
streamer = TextStreamer(tokenizer, skip_prompt=True)

# 5. COORDINAZIONE E INTERAZIONE TRA I COMPONENTI
# In questa fase assembliamo le classi create sopra nelle strutture dati richieste dal metodo generate().
# - LogitsProcessorList: implementa NoUppercaseProcessor per controllare COSA viene scritto (solo minuscolo).
# - StoppingCriteriaList: implementa StopAtPeriodCriteria per controllare QUANDO fermarsi (al primo punto).
logits_processors = LogitsProcessorList([NoUppercaseProcessor()])
stopping_criteria = StoppingCriteriaList([StopAtPeriodCriteria()])

print(f"Prompt iniziale: {prompt}")
print("Risposta (Generazione vincolata in minuscolo e streaming):")

# 6. ESECUZIONE DEL PROCESSO DI GENERAZIONE
# Il metodo generate() orchestra l'interazione tra modello, filtri e criteri di stop:
# 1. Calcola le probabilità dei token.
# 2. Applica NoUppercaseProcessor per filtrare le maiuscole.
# 3. Seleziona un token (do_sample).
# 4. Invia il token allo streamer per la visualizzazione immediata.
# 5. Interroga StopAtPeriodCriteria per decidere se continuare o fermarsi.
with torch.no_grad():
    model.generate(
        **inputs,
        streamer=streamer,                  # Visualizza il testo in tempo reale.
        logits_processor=logits_processors,     # Forza il vincolo sulle minuscole.
        stopping_criteria=stopping_criteria,     # Gestisce la chiusura automatica della frase.
        max_new_tokens=100,                 # Numero massimo di nuovi termini generabili.
        do_sample=True,                     # Introduce varietà nella risposta.
        temperature=0.7,                    # Bilancia fedeltà e creatività.
        pad_token_id=tokenizer.eos_token_id
    )

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4514.20it/s]


Prompt iniziale: Some thoughts about technology, Apple vs Microsoft:
Risposta (Generazione vincolata in minuscolo e streaming):


—


2.
